## Minimal energy path on  2d Müller-Brown potential



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import matplotlib.cm as cm
import bisect

### 2d Müller-Brown potential

We define: 

1. the potential $V$ as a class 

2. a function for sampling the trajectory of the Brownian dynamics

$$
dX_t = - \nabla V(X_t) dt + \sqrt{2\beta^{-1}} dW_t
$$

In [ ]:
# Mueller-Brown potential $V(x)$ in 2d
class MuellerPotential:
    def __init__(self, *argv):
        
        # Parameters in the definition of V
        self.a = [-1, -1, -6.5, 0.7]
        self.b = [0, 0, 11, 0.6]
        self.c = [-10, -10, -6.5, 0.7]
        self.A = [-200, -100, -170, 15]
        self.xc = [1, 0, -0.5, -1]
        self.yc = [0, 0.5, 1.5, 1]

        self.x_domain = [-1.8, 1.2]
        self.y_domain = [-0.5, 2.2]
        self.v_min_max = [-130, 20]
        self.contour_levels = [-130, -100, -80, -60, -40, -20, 0.0]
        self.density_max = 0.35
        
    # the potential    
    def V(self, x):
        s = 0
        for i in range(4):
            dx = x[0] - self.xc[i]
            dy = x[1] - self.yc[i]
            s += self.A[i] * np.exp(self.a[i] * dx**2 + self.b[i] * dx * dy + self.c[i] * dy**2)
        return s
    
    # gradient of the potential    
    def gradV(self, x):
        s = 0
        dVx = 0
        dVy = 0
        for i in range(4):
            dx = x[0] - self.xc[i]
            dy = x[1] - self.yc[i]            
            dVx += self.A[i] * (2 * self.a[i] * dx + self.b[i] * dy) * np.exp(self.a[i] * dx**2 + self.b[i] * dx * dy + self.c[i] * dy**2)
            dVy += self.A[i] * (self.b[i] * dx + 2 * self.c[i] * dy) * np.exp(self.a[i] * dx**2 + self.b[i] * dx * dy + self.c[i] * dy**2)
        return np.array((dVx, dVy))

# sample the SDE using Euler-Maruyama scheme

def sample(pot, beta=1.0, delta_t = 0.001, N=10000, seed=42):
    rng = np.random.default_rng(seed=seed)
     
    X = [-0.6, 1.2]
    dim = 2 
    traj = []
    save = 100
    tlist = []
    for i in tqdm(range(N)):
        b = rng.normal(size=(dim,))
        X = X - pot.gradV(X) * delta_t + np.sqrt(2 * delta_t/beta) * b
        if i % save==0:
            traj.append(X)
            tlist.append(i * delta_t)

    return np.array(tlist), np.array(traj)

### define an object of the potential class

In [ ]:
pot = MuellerPotential()  

### discrete the 2d space into a grid 

In [ ]:
nx = 100
ny = 150

dx = (pot.x_domain[1] - pot.x_domain[0]) / nx
dy = (pot.y_domain[1] - pot.y_domain[0]) / ny

gridx = np.linspace(pot.x_domain[0], pot.x_domain[1], nx)
gridy = np.linspace(pot.y_domain[0], pot.y_domain[1], ny)
x_plot = np.outer(gridx, np.ones(ny)) 
y_plot = np.outer(gridy, np.ones(nx)).T 

# get grid points
x2d = np.concatenate((x_plot.reshape(nx * ny, 1), y_plot.reshape(nx * ny, 1)), axis=1)

# compute the potential $V$ at grid points
pot_on_grid = np.array([pot.V(x) for x in x2d]).reshape(nx, ny)

### visualize the potential

The potenial has two deep local minimum points, which are separated by a shallow local minimum point.

In [ ]:
fig, ax = plt.subplots(figsize=(5,5))

# visualize the potential and its contour lines
im = ax.pcolormesh(x_plot, y_plot, pot_on_grid, cmap='coolwarm', vmin=pot.v_min_max[0], vmax=pot.v_min_max[1])
contours = ax.contour(x_plot, y_plot, pot_on_grid,  pot.contour_levels)

# add potential value to the contour lines
ax.clabel(contours, inline=True, fontsize=13,colors='black')

ax.set_aspect('equal')
ax.tick_params(axis='both', labelsize=15)

ax.set_xticks([-1.5, -1.0, -0.5, 0, 0.5, 1.0])
ax.set_yticks([-0.5, 0, 0.5, 1.0, 1.5, 2.0])
ax.set_xlim([pot.x_domain[0], pot.x_domain[1]])
ax.set_ylim([pot.y_domain[0], pot.y_domain[1]])

ax.set_title("Müller-Brown potential",fontsize=20)
cbar = fig.colorbar(im, ax=ax, shrink=0.7)
cbar.ax.tick_params(labelsize=15)

### generate a long trajectory of the Brownian dynamics

$$
dX_t = - \nabla V(X_t) dt + \sqrt{2\beta^{-1}} dW_t
$$

The trajectory data is not used in this notebook. But it is helpful to understand the dynamics.  

In [ ]:
tlist, trajectory = sample(pot, beta=0.1, delta_t=0.0002, N=1000000)

print ('shape of the trajectory data:', trajectory.shape)

### plot the trajectory data

It can be seen that the transitions happen very rarely.

In [ ]:
fig = plt.figure(figsize=(12,3))

ax1 = fig.add_subplot(1, 3, 1)
ax2 = fig.add_subplot(1, 3, 2)
ax3 = fig.add_subplot(1, 3, 3)

# evaluate potential on grid points
pot_on_grid = np.array([pot.V(x) for x in x2d]).reshape(nx, ny)
# plot contour lines of the potential
contours = ax1.contour(x_plot, y_plot, pot_on_grid, levels=pot.contour_levels, cmap='coolwarm')

# scatter plot of the trajectory data
ax1.scatter(trajectory[:,0], trajectory[:,1], alpha=0.5, c='k', s=4)

ax1.set_xlim([pot.x_domain[0], pot.x_domain[1]])
ax1.set_ylim([pot.y_domain[0], pot.y_domain[1]])
ax1.set_title('trajectory')

# plot time-series of the x component
ax2.plot(tlist, trajectory[:,0])
ax2.set_ylim([pot.x_domain[0], pot.x_domain[1]])
ax2.set_title('x coodinate along trajectory')

# plot time-series of the y component
ax3.plot(tlist, trajectory[:,1])
ax3.set_ylim([pot.y_domain[0], pot.y_domain[1]])
ax3.set_title('y coodinate along trajectory')

plt.show()

### Compute the minimal energy path (MEP) using string method

**MEP**: $\hspace{1cm}\dot{\varphi}(t) \parallel \nabla V(\varphi(t))$. 
    
**Input**:

1. Two local minimum points $a$ and $b$. 
2. Initial path $\varphi(s)$, where $s\in[0,1]$, such that $\varphi(0) = a$ and $\varphi(1)=b$.


**Algorithm**:

1. Update: 
 $\varphi_i^{*} = -\tau \nabla V(\varphi_i^k) + \varphi_i^k\hspace{0.5cm}$,  for $i = 0,1,2,\cdots, N$


2. Reparametrization. Obtain new state $\varphi_i^{k+1}$ by linear interpolation: 

$$
  \varphi_{i}^{k+1} = \frac{L_{j+1}^{*}-L_i}{L_{j+1}^{*} - L_j^*} \varphi_j^* + \frac{L_i - L_{j}^{*}}{L_{j+1}^* - L_j^*} \varphi_{j+1}^*\,,
$$
 where $L_j^* < L_i \leq L_{j+1}^*$, $L_0^* = L_0 = 0$, and
    \begin{equation*}
       L_i^* = \frac{\sum\limits_{j=0}^{i-1} |\varphi_{j+1}^{*} - \varphi_{j}^{*}|}{ \sum\limits_{j=0}^{N-1} |\varphi_{j+1}^{*} -
      \varphi_{j}^{*}|}\,, \quad L_i = \frac{i}{N}, \quad\, i = 1,\cdots, N\,. 
    \end{equation*}

In [ ]:
    
def find_MEP(pot, N, min_A, min_B, dt=0.0001, NSteps=1000, with_reparam=True):
    
    # initial path is a straight line connecting a and b 
    line = np.array(np.linspace(min_A, min_B, N)) 
    
    delta_t = dt

    for istep in range(NSteps):
        
        # Update: evolve each state on the path   
        for idx in range(N) :
            X = line[idx, :]
            line[idx,:] = X - pot.gradV(X) * delta_t 

        # Compute distances of adjacent states 
        dists = [0.0]
        for idx in range(N-1) :
            dists.append( np.linalg.norm(line[idx,:] - line[idx+1,:]) + dists[idx] )
        
        # normalize the total distance
        dists = dists / dists[-1]

        # Reparametrization

        new_states = [line[0,:]]
        
        for idx in range(N-2):
            r = (idx+1) / (N - 1)
            # find the index 
            pos = bisect.bisect_left(dists, r)
            # obtain new state by linear interpolation
            t = (r-dists[pos-1]) / (dists[pos] - dists[pos-1])                            
            new_states.append((1-t) * line[pos-1, :] + t * line[pos, :])

        new_states.append(line[-1,:])
        
        if with_reparam : 
            line = np.array(new_states)

    return line

### visualize the path computed by string method 

In [ ]:
def plot_potential_on_axis(ax):
    
    # visualize the potential and its contour lines
    im = ax.pcolormesh(x_plot, y_plot, pot_on_grid, cmap='coolwarm', vmin=pot.v_min_max[0], vmax=pot.v_min_max[1])
    contours = ax.contour(x_plot, y_plot, pot_on_grid,  pot.contour_levels)

    ax.set_aspect('equal')
    ax.tick_params(axis='both', labelsize=10)

    ax.set_xticks([-1.5, -1.0, -0.5, 0, 0.5, 1.0])
    ax.set_yticks([-0.5, 0, 0.5, 1.0, 1.5, 2.0])
    ax.set_xlim([pot.x_domain[0], pot.x_domain[1]])
    ax.set_ylim([pot.y_domain[0], pot.y_domain[1]])    
        
fig = plt.figure(figsize=(9,9))

ax1 = fig.add_subplot(2, 2, 1)
ax2 = fig.add_subplot(2, 2, 2)
ax3 = fig.add_subplot(2, 2, 3)
ax4 = fig.add_subplot(2, 2, 4)

# two local minimal points a and b
xa = [-0.5, 1.5]
xb = [0.6, 0.0]

# get the initial path (NSteps=0)
path = find_MEP(pot, 30, xa, xb, NSteps=0)
plot_potential_on_axis(ax=ax1)
# plot the path
ax1.scatter(path[:,0], path[:,1], c='k')
ax1.set_title('step 0')

# compute the path after 20 iterations.
path = find_MEP(pot, 30, xa, xb, NSteps=20)
plot_potential_on_axis(ax=ax2)
# plot the path
ax2.scatter(path[:,0], path[:,1], c='k')
ax2.set_title('step 20')

# compute the path after 60 iterations.
path = find_MEP(pot, 30, xa, xb, NSteps=60)
plot_potential_on_axis(ax=ax3)
ax3.scatter(path[:,0], path[:,1], c='k')
ax3.set_title('step 60')

# compute the path after 100 iterations.
path = find_MEP(pot, 30, xa, xb, NSteps=100)
plot_potential_on_axis(ax=ax4)
ax4.scatter(path[:,0], path[:,1], c='k')
ax4.set_title('step 100')

plt.show()

### the effect of reparametrization

Recompute the path by string method, but without reparametrization.

It can be seen that the discrete states converge to the corresponding local minimal states.

Hence, in order to have a good representation of path, reparametrization is necessary.

In [ ]:
fig = plt.figure(figsize=(9,9))

ax1 = fig.add_subplot(2, 2, 1)
ax2 = fig.add_subplot(2, 2, 2)
ax3 = fig.add_subplot(2, 2, 3)
ax4 = fig.add_subplot(2, 2, 4)

xa = [-0.5, 1.5]
xb = [0.6, 0.0]

path = find_MEP(pot, 30, xa, xb, NSteps=0, with_reparam=False)
plot_potential_on_axis(ax=ax1)
ax1.scatter(path[:,0], path[:,1], c='k')
ax1.set_title('step 0')

path = find_MEP(pot, 30, xa, xb, NSteps=20, with_reparam=False)
plot_potential_on_axis(ax=ax2)
ax2.scatter(path[:,0], path[:,1], c='k')
ax2.set_title('step 20')

path = find_MEP(pot, 30, xa, xb, NSteps=60, with_reparam=False)
plot_potential_on_axis(ax=ax3)
ax3.scatter(path[:,0], path[:,1], c='k')
ax3.set_title('step 60')

path = find_MEP(pot, 30, xa, xb, NSteps=100, with_reparam=False)
plot_potential_on_axis(ax=ax4)
ax4.scatter(path[:,0], path[:,1], c='k')
ax4.set_title('step 100')

plt.show()

### recompute the minimal energy path

1. Compute the state with the highest potential along the path.
2. Estimate the tangent direction of the path 

In [ ]:
# local minimal points a and b
xa = [-0.5, 1.5]
xb = [0.6, 0.0]

# evolve the path for 200 steps
path = find_MEP(pot, 30, xa, xb, NSteps=200)

# potential along the path
pot_on_line = [pot.V(state) for state in path]

# state with the highest potential
max_idx = np.argmax(pot_on_line)
X = path[max_idx,:]

# estimate tangent direction by finite difference 
tau = path[max_idx+1] - path[max_idx-1]
tau = tau / np.linalg.norm(tau)

print ('tangent direction:', tau)

### Plot: 

1. the state on the path with the higest potential
2. and the potential along the path

In [ ]:
fig = plt.figure(figsize=(9,4))
ax1 = fig.add_subplot(1, 2, 1)

plot_potential_on_axis(ax=ax1)
ax1.scatter(path[:,0], path[:,1], c='k')
ax1.set_title('MEP')

# plot the state with highest potential
ax1.plot(path[max_idx,0], path[max_idx,1], marker='o', c='w', markersize=12)

ax2 = fig.add_subplot(1, 2, 2)

# plot the potential along the path
ax2.plot(pot_on_line)
ax2.set_title('Potential along MEP')
plt.show()


### Saddle point is unstable under the ODE 
$$\frac{dX_t}{dt} = -\nabla V(X_t).$$

Starting from the saddle point, the ODE above will converge to a local minimal state.

In [ ]:
max_idx = np.argmax(pot_on_line)
X = path[max_idx,:]

X_list = [X]
N = 1000
delta_t = 0.0001

# simulate the ODE for 1000 steps.
for idx in range(N) :
    X = X - pot.gradV(X) * delta_t 
    X_list.append(X)
    
fig, ax = plt.subplots()

plot_potential_on_axis(ax=ax)

X_traj = np.array(X_list)

# plot the trajectory of ODE 
ax.scatter(X_traj[::5,0], X_traj[::5,1])

ax.plot(X_traj[0,0], X_traj[0,1], marker='o', c='w', markersize=12)
ax.plot(X_traj[-1,0], X_traj[-1,1], marker='X', c='k', markersize=6)

plt.show()    

### Climbing image method to refine the saddle point. 

**Require**:
1. Initial guess of saddle point.
2. Estimation of unit tangent direction.

**Dynamics**: $\quad \dot{\phi_t} = -\nabla V(\phi_t) + 2 \langle\nabla V(\phi_t), \tau_0\rangle\tau_0$.

In [ ]:
# initial guess
X = path[max_idx,:]

X_list = [X]
N = 1000
delta_t = 0.0001

for idx in range(N) :
    drift = pot.gradV(X)
    X = X + (- drift + 2.0 * np.dot(drift, tau) * tau) * delta_t
    X_list.append(X)

### Visualize the trajectory of the above dynamics 

It can be seen that the saddle point becomes stable. 

In [ ]:
fig, ax = plt.subplots()

plot_potential_on_axis(ax=ax)

X_traj = np.array(X_list)
ax.scatter(X_traj[::5,0], X_traj[::5,1])

ax.plot(X_traj[0,0], X_traj[0,1], marker='o', c='w', markersize=12)
ax.plot(X_traj[-1,0], X_traj[-1,1], marker='X', c='k', markersize=6)

plt.show()

print ("Initial guess (white):", X_traj[0], "\npotential:", pot.V(X_traj[0]))
print ("\nFinal state (black):", X_traj[-1], "\npotential:", pot.V(X_traj[-1]))

### Repeat the experiment for the other saddle point

The second saddle point recorresponds to the second peak of potential.

In [ ]:
# state at the second potential peak
second_max_idx = np.argmax(pot_on_line[20:]) + 20

X = path[second_max_idx,:]

# estimate tangent direction by finite difference 
tau = path[second_max_idx+1] - path[second_max_idx-1]
tau = tau / np.linalg.norm(tau)

print ('tangent direction:', tau)

### Again, the saddle point is unstable under the ODE
$$\frac{dX_t}{dt} = -\nabla V(X_t).$$

The initial guess of the saddle point converges to the local minimum point in the middle.

In [ ]:
X = path[second_max_idx,:]
X_list = [X]
N = 1000
delta_t = 0.0001

# simulate the ODE for 1000 steps.
for idx in range(N) :
    X = X - pot.gradV(X) * delta_t 
    X_list.append(X)
    
fig, ax = plt.subplots()

plot_potential_on_axis(ax=ax)

X_traj = np.array(X_list)

# plot the trajectory of ODE 
ax.scatter(X_traj[::5,0], X_traj[::5,1])

ax.plot(X_traj[0,0], X_traj[0,1], marker='o', c='w', markersize=12)
ax.plot(X_traj[-1,0], X_traj[-1,1], marker='X', c='k', markersize=6)

plt.show()    

### Let's refine the saddle point using the modified ODE: 
 $\quad \dot{\phi_t} = -\nabla V(\phi_t) + 2 \langle\nabla V(\phi_t), \tau_0\rangle\tau_0$.

In [ ]:
# initial guess
X = path[second_max_idx,:]

X_list = [X]
N = 1000
delta_t = 0.0001

for idx in range(N) :
    drift = pot.gradV(X)
    X = X + (- drift + 2.0 * np.dot(drift, tau) * tau) * delta_t
    X_list.append(X)

### Check that the dynamics converges to the saddle point.

In [ ]:
fig, ax = plt.subplots()

plot_potential_on_axis(ax=ax)

X_traj = np.array(X_list)
ax.scatter(X_traj[::5,0], X_traj[::5,1])

ax.plot(X_traj[0,0], X_traj[0,1], marker='o', c='w', markersize=12)
ax.plot(X_traj[-1,0], X_traj[-1,1], marker='X', c='k', markersize=6)

plt.show()

print ("Initial guess (white):", X_traj[0], "\npotential:", pot.V(X_traj[0]))
print ("\nFinal state (black):", X_traj[-1], "\npotential:", pot.V(X_traj[-1]))